# LabelMe 标注检查

这个 notebook 会把原图标注重新画回去，并生成 crop 拼图，方便检查原始标注和裁剪效果。


## 0. 依赖导入


In [ ]:
from pathlib import Path
import json

from PIL import Image, ImageDraw

print('✓ 依赖导入完成')


## 1. 核心函数


In [ ]:
def _normalize_shape_bbox(points):
    if not points or len(points) < 2:
        return None
    x_values = [point[0] for point in points]
    y_values = [point[1] for point in points]
    return [int(min(x_values)), int(min(y_values)), int(max(x_values)), int(max(y_values))]


def load_annotations(annotations_dir):
    annotations = []
    for annotation_path in sorted(Path(annotations_dir).glob('*.json')):
        payload = json.loads(annotation_path.read_text(encoding='utf-8'))
        metadata = payload.get('metadata', {})
        original_path = metadata.get('original_path') or payload.get('imagePath')
        instances = metadata.get('instances', [])

        if instances:
            for instance in instances:
                annotations.append({
                    **instance,
                    'annotation_path': str(annotation_path),
                    'original_path': instance.get('original_path') or original_path,
                })
            continue

        for index, shape in enumerate(payload.get('shapes', []), start=1):
            if shape.get('shape_type', 'rectangle') != 'rectangle':
                continue
            bbox = _normalize_shape_bbox(shape.get('points', []))
            if not bbox:
                continue
            annotations.append({
                'id': shape.get('id', index),
                'class_name': shape.get('label'),
                'bbox': bbox,
                'annotation_path': str(annotation_path),
                'original_path': original_path,
            })

    return annotations


def _resolve_image_path(source_images_dir, original_path):
    original_path = Path(original_path)
    candidate = Path(source_images_dir) / original_path
    if candidate.exists():
        return candidate
    return Path(source_images_dir) / original_path.name


def _apply_padding(bbox, width, height, padding_ratio):
    x1, y1, x2, y2 = bbox
    box_width = max(x2 - x1, 1)
    box_height = max(y2 - y1, 1)
    pad_x = box_width * padding_ratio
    pad_y = box_height * padding_ratio
    return [
        max(int(x1 - pad_x), 0),
        max(int(y1 - pad_y), 0),
        min(int(x2 + pad_x), width),
        min(int(y2 + pad_y), height),
    ]


def _group_instances_by_image(instances):
    grouped = {}
    for instance in instances:
        grouped.setdefault(instance['original_path'], []).append(instance)
    return grouped


def _draw_overlay(image, instances):
    canvas = image.copy()
    draw = ImageDraw.Draw(canvas)
    for instance in instances:
        x1, y1, x2, y2 = instance['bbox']
        label = instance.get('class_name', 'unknown')
        color = '#ef4444' if label == 'abnormal' else '#22c55e'
        draw.rectangle([x1, y1, x2, y2], outline=color, width=6)
        draw.text((x1, max(0, y1 - 24)), label, fill=color)
    return canvas


def _fit_crop_to_tile(crop, tile_size=240, background_color='white'):
    canvas = Image.new('RGB', (tile_size, tile_size), background_color)
    fitted = crop.copy()
    fitted.thumbnail((tile_size, tile_size))
    offset_x = (tile_size - fitted.width) // 2
    offset_y = (tile_size - fitted.height) // 2
    canvas.paste(fitted, (offset_x, offset_y))
    return canvas


def _build_crop_sheet(image, instances, preview_limit=12, padding_ratio=0.05):
    preview_instances = instances[:preview_limit]
    if not preview_instances:
        return Image.new('RGB', (400, 200), 'white')

    tile_size = 240
    header_height = 28
    cols = min(4, len(preview_instances))
    rows = (len(preview_instances) + cols - 1) // cols
    sheet = Image.new('RGB', (cols * tile_size, rows * (tile_size + header_height)), 'white')

    for index, instance in enumerate(preview_instances):
        padded_bbox = _apply_padding(instance['bbox'], image.width, image.height, padding_ratio)
        x1, y1, x2, y2 = padded_bbox
        crop = _fit_crop_to_tile(image.crop((x1, y1, x2, y2)), tile_size=tile_size)
        offset_x = (index % cols) * tile_size
        offset_y = (index // cols) * (tile_size + header_height)
        sheet.paste(crop, (offset_x, offset_y + header_height))

        draw = ImageDraw.Draw(sheet)
        label = instance.get('class_name', 'unknown')
        draw.text((offset_x + 8, offset_y + 4), f"#{instance.get('id', index + 1)} {label}", fill='black')

    return sheet


def create_review_outputs(
    annotations_dir,
    source_images_dir,
    output_dir,
    preview_limit=12,
    padding_ratio=0.05,
):
    instances = load_annotations(annotations_dir)
    grouped = _group_instances_by_image(instances)

    output_dir = Path(output_dir)
    overlays_dir = output_dir / 'overlays'
    crop_sheets_dir = output_dir / 'crop_sheets'
    overlays_dir.mkdir(parents=True, exist_ok=True)
    crop_sheets_dir.mkdir(parents=True, exist_ok=True)

    processed_images = 0

    for original_path, image_instances in grouped.items():
        image_path = _resolve_image_path(source_images_dir, original_path)
        if not image_path.exists():
            continue

        with Image.open(image_path) as image:
            rgb_image = image.convert('RGB')
            stem = Path(original_path).stem
            _draw_overlay(rgb_image, image_instances).save(overlays_dir / f"{stem}_overlay.jpg", quality=90)
            _build_crop_sheet(rgb_image, image_instances, preview_limit=preview_limit, padding_ratio=padding_ratio).save(
                crop_sheets_dir / f"{stem}_crops.jpg",
                quality=90,
            )
            processed_images += 1

    summary = {
        'processed_images': processed_images,
        'reviewed_instances': len(instances),
        'output_dir': str(output_dir),
    }

    (output_dir / 'review_summary.json').write_text(
        json.dumps(summary, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )
    return summary


print('✓ 核心函数准备完成')


## 2. 路径与参数配置


In [ ]:
ANNOTATIONS_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets/data_annotated_2class')
SOURCE_IMAGES_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/datasets/data_annotated_2class')
OUTPUT_DIR = Path('/home/hujing/yolo-web-ui/offline_workspace/outputs/labelme_review')

PREVIEW_LIMIT = 12
PADDING_RATIO = 0.05

print('ANNOTATIONS_DIR =', ANNOTATIONS_DIR)
print('SOURCE_IMAGES_DIR =', SOURCE_IMAGES_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)


## 3. 生成检查结果


In [ ]:
summary = create_review_outputs(
    annotations_dir=ANNOTATIONS_DIR,
    source_images_dir=SOURCE_IMAGES_DIR,
    output_dir=OUTPUT_DIR,
    preview_limit=PREVIEW_LIMIT,
    padding_ratio=PADDING_RATIO,
)

print(json.dumps(summary, indent=2, ensure_ascii=False))


## 4. 输出目录提示


In [ ]:
print('overlay 目录:')
print(OUTPUT_DIR / 'overlays')
print('\ncrop_sheet 目录:')
print(OUTPUT_DIR / 'crop_sheets')
